In [1]:
import os
from dotenv import load_dotenv
from azure.ai.evaluation._model_configurations import AzureOpenAIModelConfiguration


# Load from .env file
load_dotenv()

def get_required_env_var(name: str) -> str:
    value = os.getenv(name)
    if not value:
        raise ValueError(
            f"{name} environment variable is not set. Please set it in your .env file."
        )
    return value

endpoint = get_required_env_var("AZURE_OPENAI_ENDPOINT")
api_key = get_required_env_var("AZURE_OPENAI_API_KEY")
deployment_name = get_required_env_var("AZURE_OPENAI_DEPLOYMENT_NAME")
brand_name = get_required_env_var("BRAND_NAME")
restaurant_brand = get_required_env_var("RESTAURANT_BRAND")
api_version = "2024-12-01-preview"
conversation_style = get_required_env_var("CONVERSATION_STYLE")

print(f"Evaluating: {restaurant_brand} with {conversation_style} style")

model_config = {
    "azure_endpoint": endpoint,
    "azure_deployment": deployment_name,
    "api_key": api_key,
    "api_version": api_version
}


Evaluating: contoso_restaurant with default style


In [2]:
deployment_name

'gpt-4.1'

In [5]:
import json
from pathlib import Path
from typing import Any, Dict, List, Optional
import sys 

# Load brand configuration for grounding context
def load_brand_grounding_data():
    """Load brand-specific menu and personality data for grounding evaluation."""
    
    # Load brand configuration
    brand_config_path = Path("../streaming_ordering_chatbot/resources/brand_config_files.json")
    with open(brand_config_path, "r") as f:
        brand_configs = json.load(f)
    
    selected_brand = brand_configs.get(brand_name, {})
    
    menu_config_path = os.getenv("MENU_CONFIG_PATH")   #Path("../streaming_ordering_chatbot/api/flows/data/contoso_restaurant_menu.json")
    if menu_config_path:
        menu_path = Path(menu_config_path)
    else:
        menu_path = Path("../streaming_ordering_chatbot/api/flows/data/contoso_restaurant_menu.json")
    with open(menu_path, "r") as f:
        menu_configs = json.load(f)

    # Use menu_configs directly since it contains the menu structure
    brand_menu = menu_configs
    # Create ordering-specific grounding scenarios
    ordering_grounding_data = [
        {
            "content": "What menu items do you have?",
            "context": f"""
            "Brand": {selected_brand.get('name')},
            "Tone": {selected_brand.get('tone')},
            "Style": {selected_brand.get('style')},
            "Values": {', '.join(selected_brand.get('values',[]))}
            
            Menu Items:
            "Brand": {brand_menu.get('brand_info', {}).get('name')},
            "menu_items":{brand_menu.get('menu_items')},
            "toppings": {brand_menu.get('toppings')},
            "item_types": {brand_menu.get('item_types')},
            "amount_codes": {brand_menu.get('amount_codes')},

            Response Guidelines:
            Use {selected_brand.get('tone')} tone, {selected_brand.get('style')} style, and emphasize some {', '.join(selected_brand.get('key_phrases', []))} key phrases and some {', '.join(selected_brand.get('values', []))} brand values in responses.
            Ordered items should be in the Menu Items: The menu is {brand_menu.get('menu_items')} menu items, toppings available are {brand_menu.get('toppings')}, item types include {brand_menu.get('item_types')}, and amount codes are {brand_menu.get('amount_codes')}.
            """
        },
        {
            "content": "I want to order a burger with fries",
            "context": f"""
            Current Order: Empty
            
            Ordering Process:
            1. Acknowledge the order request
            2. Confirm specific order type and items
            3. Validate items against menu
            4. Add items to order
            5. Provide order summary
            6. Encourage cutomer to add order to cart
            
            "Brand": {selected_brand.get('name')},
            "Tone": {selected_brand.get('tone')},
            "Style": {selected_brand.get('style')},
            "Values": {', '.join(selected_brand.get('values', []))}
            
            Menu Items:
            "Brand": {brand_menu.get('brand_info', {}).get('name')},
            "menu_items":{brand_menu.get('menu_items')},
            "toppings": {brand_menu.get('toppings')},
            "item_types": {brand_menu.get('item_types')},
            "amount_codes": {brand_menu.get('amount_codes')},

            Response Guidelines:
            Use {selected_brand.get('tone')} tone, {selected_brand.get('style')} style, and emphasize some {', '.join(selected_brand.get('key_phrases', []))} key phrases and some {', '.join(selected_brand.get('values', []))} brand values in responses.
            Ordered items should be in the Menu Items: The menu is {brand_menu.get('menu_items')} menu items, toppings available are {brand_menu.get('toppings')}, item types include {brand_menu.get('item_types')}, and amount codes are {brand_menu.get('amount_codes')}.
            """
        },
        {
            "content": "I want preztel hambugger, 2 seasame cheeseburger, 1 unsalted fries, and 2 large sprites. Give me my order summary",
            "context": f"""
             Menu Items:
            "Brand": {brand_menu.get('brand_info', {}).get('name')},
            "menu_items":{brand_menu.get('menu_items')},
            "toppings": {brand_menu.get('toppings')},
            "item_types": {brand_menu.get('item_types')},
            "amount_codes": {brand_menu.get('amount_codes')},
            
            Response Guidelines: Provide clear order summary, validate order against menu items, and maintain {selected_brand.get('tone')} tone, {selected_brand.get('style')} style, and emphasize some {', '.join(selected_brand.get('key_phrases'))} key phrases and some {', '.join(selected_brand.get('values'))} brand values in responses.
              """
        },
        {
            "content": "I want 2 burritos and vegetable spring roll",
            "context": f"""
            Current Order: Empty
            
            Ordering Process:
            1. Acknowledge the order request
            2. Confirm specific order type and items
            3. Validate items against menu
            4. Add items to order
            5. Provide order summary
            6. Encourage cutomer to add order to cart
            
            "Brand": {selected_brand.get('name')},
            "Tone": {selected_brand.get('tone')},
            "Style": {selected_brand.get('style')},
            "Values": {', '.join(selected_brand.get('values', []))}
            
            Menu Items:
            "Brand": {brand_menu.get('brand_info', {}).get('name')},
            "menu_items":{brand_menu.get('menu_items')},
            "toppings": {brand_menu.get('toppings')},
            "item_types": {brand_menu.get('item_types')},
            "amount_codes": {brand_menu.get('amount_codes')}

            Response Guidelines:
            Use {selected_brand.get('tone')} tone, {selected_brand.get('style')} style, and emphasize some {', '.join(selected_brand.get('key_phrases', []))} key phrases and some {', '.join(selected_brand.get('values', []))} brand values in responses.
            Ordered items should be in the Menu Items: The menu is {brand_menu.get('menu_items')} menu items, toppings available are {brand_menu.get('toppings')}, item types include {brand_menu.get('item_types')}, and amount codes are {brand_menu.get('amount_codes')}.
            If ordered items are not available, suggest alternatives from the menu.
            """
        },
        {
            "content": "Can I get a vegetarian pizza with extra cheese? Also include a preztel hamburger and 2 large sprites, do you have a spicy sauce in your toppings? Add it to my preztel hamburger. Also give me 5 medium fries and a cheeseburger with no toppings.",
            "context": f"""
            Current Order: Empty
            
            Ordering Process:
            1. Acknowledge the order request
            2. Confirm specific order type and items
            3. Validate items against menu
            4. suggest alternatives for unavailable items
            4. Add items to order
            5. Provide order summary
            6. Encourage cutomer to add order to cart
            
            "Brand": {selected_brand.get('name')},
            "Tone": {selected_brand.get('tone')},
            "Style": {selected_brand.get('style')},
            "Values": {', '.join(selected_brand.get('values', []))}
            
            Menu Items:
            "Brand": {brand_menu.get('brand_info', {}).get('name')},
            "menu_items":{brand_menu.get('menu_items')},
            "toppings": {brand_menu.get('toppings')},
            "item_types": {brand_menu.get('item_types')},
            "amount_codes": {brand_menu.get('amount_codes')}

            Response Guidelines:
            Use {selected_brand.get('tone')} tone, {selected_brand.get('style')} style, and emphasize some {', '.join(selected_brand.get('key_phrases', []))} key phrases and some {', '.join(selected_brand.get('values', []))} brand values in responses.
            Ordered items should be in the Menu Items: The menu is {brand_menu.get('menu_items')} menu items, toppings available are {brand_menu.get('toppings')}, item types include {brand_menu.get('item_types')}, and amount codes are {brand_menu.get('amount_codes')}.
            If ordered items are not available, suggest alternatives from the menu.
            """
        },
        {
            "content": "Remove 1 of the preztel hamburger and add 2 sesame cheeseburger with 1 salted fries and 1 unsalted fries.",
            "current_order": {"items": [{"name": "hamburger", "buns": "preztel", "quantity": 3}]},
            "context": f"""
            Current Order: {json.dumps("current_order")}

            Ordering Process:
            1. Acknowledge the order request
            2. Confirm specific order type and items
            3. Validate items against menu
            4. Modify and add items to order
            5. Provide order summary
            6. Encourage cutomer to add order to cart
            
            "Brand": {selected_brand.get('name')},
            "Tone": {selected_brand.get('tone')},
            "Style": {selected_brand.get('style')},
            "Values": {', '.join(selected_brand.get('values', []))}
            
            Menu Items:
            "Brand": {brand_menu.get('brand_info', {}).get('name')},
            "menu_items":{brand_menu.get('menu_items')},
            "toppings": {brand_menu.get('toppings')},
            "item_types": {brand_menu.get('item_types')},
            "amount_codes": {brand_menu.get('amount_codes')}

            Response Guidelines:
            Use {selected_brand.get('tone')} tone, {selected_brand.get('style')} style, and emphasize some {', '.join(selected_brand.get('key_phrases', []))} key phrases and some {', '.join(selected_brand.get('values', []))} brand values in responses.
            Ordered items should be in the Menu Items: The menu is {brand_menu.get('menu_items')} menu items, toppings available are {brand_menu.get('toppings')}, item types include {brand_menu.get('item_types')}, and amount codes are {brand_menu.get('amount_codes')}.
            If ordered items are not available, suggest alternatives from the menu.
            """
        }
    ]
    
    return ordering_grounding_data

# Load the grounding data
ordering_data = load_brand_grounding_data()
conversation_turns = [[item] for item in ordering_data]

print(f"Loaded {len(conversation_turns)} ordering scenarios for groundedness evaluation")


Loaded 6 ordering scenarios for groundedness evaluation


In [6]:
conversation_turns

[[{'content': 'What menu items do you have?',
   'context': '\n            "Brand": Contoso Restaurant,\n            "Tone": warm and welcoming,\n            "Style": Like a friendly neighborhood diner but not chatty. Use warm, inviting language. Focus on comfort and quality. Highlight personal service and home-style cooking.,\n            "Values": quality, community engagement, comfort, customer satisfaction, sustainability\n\n            Menu Items:\n            "Brand": Contoso Restaurant,\n            "menu_items":{\'hamburger\': {\'code_pattern\': \'HB{size}{patties}{bun}{cook}\', \'category\': \'burger\', \'sizes\': [\'14\', \'13\'], \'patties\': [\'S1\', \'D\'], \'buns\': [\'S\', \'P\'], \'cooks\': [\'N\', \'W\'], \'default\': {\'size\': \'14\', \'patties\': \'S\', \'buns\': \'S\', \'cook\': \'N\'}, \'toppings\': [\'L\', \'T\', \'O\', \'C\', \'M\', \'Y\', \'R\', \'P\', \'B\'], \'name_variations\': [\'hamburger\', \'burger\']}, \'cheeseburger\': {\'code_pattern\': \'CZHB{size}{p

In [ ]:
import sys 
sys.path.append('../')

from streaming_ordering_chatbot.api.flows.classification_flow_SK import OrderIntentFlowSK 
from streaming_ordering_chatbot.api.flows.conversation_flows_SK import PreambleFlowSK, OrderAssistantFlowSK, SummaryFlowSK 
from streaming_ordering_chatbot.api.flows.order_flow_SK import OrderFlowSK 
from streaming_ordering_chatbot.api.models import Message 
from azure.identity import DefaultAzureCredential, get_bearer_token_provider 
from openai import AzureOpenAI

async def end_to_end_ordering_application(query: str, context: str) -> str:
        
    """ Complete end-to-end ordering application that processes user queries through ALL flows: 
     Preamble → Intent Classification → Conversation → Order Processing → Summary """ 
    try: # Initialize ALL flows 
        #preamble_flow = PreambleFlowSK(endpoint, api_key, deployment_name, restaurant_brand, conversation_style) 
        intent_flow = OrderIntentFlowSK(endpoint, api_key, deployment_name, restaurant_brand)
        conversation_flow = OrderAssistantFlowSK(endpoint, api_key, deployment_name, restaurant_brand, conversation_style)
        order_flow = OrderFlowSK(endpoint, api_key, deployment_name, restaurant_brand) 
        summary_flow = SummaryFlowSK(endpoint, api_key, deployment_name, restaurant_brand, conversation_style)

        # Prepare initial state
        chat_history = [Message(role="user", content=query)]
        current_order = {"items": []}
        flow_responses = {}
         
        # Step 2: Intent Classification
        try:
            intent_result = await intent_flow(chat_history, current_order)
            flow_responses["intent"] = intent_result
        except Exception as intent_error:
            print(f"Intent classification error: {intent_error}")
            flow_responses["intent"] = "conversation"  # Default to conversation
            intent_result = "conversation"
        
        # Step 3: Conversation Flow (with context grounding)
        try:
            # Add context as system knowledge
            enhanced_history = [
                Message(role="system", content=f"Restaurant Context: {context}"),
                Message(role="user", content=query)
            ]
            
            # Generate conversation response chunks
            conversation_chunks = []
            async for chunk in conversation_flow(enhanced_history, current_order):
                if chunk:
                    conversation_chunks.append(chunk)
            
            conversation_response = "".join(conversation_chunks)
            flow_responses["conversation"] = conversation_response
            
            # Update chat history
            chat_history.append(Message(role="assistant", content=conversation_response))
            
        except Exception as conversation_error:
            print(f"Conversation flow error: {conversation_error}")
            conversation_response = f"I understand you're asking about: {query}. Let me help you with that."
            flow_responses["conversation"] = conversation_response
        
        # Step 4: Order Processing (if order-related)
        order_response = ""
        if "order" in intent_result.lower() or any(keyword in query.lower() for keyword in ['want', 'order', 'get', 'buy', 'give', 'take']):
            try:
                order_chunks = []
                async for chunk in order_flow(enhanced_history, current_order):
                    if chunk:
                        order_chunks.append(chunk)
                
                order_response = "".join(order_chunks)
                flow_responses["order"] = order_response
                
                # Update chat history with order details
                chat_history.append(Message(role="assistant", content=f"Order processed: {order_response}"))
                
            except Exception as order_error:
                print(f"Order flow error: {order_error}")
                order_response = "I'll help you process your order."
                flow_responses["order"] = order_response
        
        # Step 5: Summary Flow (if there's an order or complex interaction)
        summary_response = ""
        if current_order.get("items") or len(chat_history) > 3:
            try:
                summary_chunks = []
                async for chunk in summary_flow(chat_history, current_order):
                    if chunk:
                        summary_chunks.append(chunk)
                
                summary_response = "".join(summary_chunks)
                flow_responses["summary"] = summary_response
                
            except Exception as summary_error:
                print(f"Summary flow error: {summary_error}")
                summary_response = "Thank you for visiting us today!"
                flow_responses["summary"] = summary_response
        
        # Combine all responses appropriately
        final_parts = []
        
        # Include conversation response
        if flow_responses.get('conversation'):
            final_parts.append(f"Response: {flow_responses['conversation']}")
        
        # Include order details if present
        if order_response:
            final_parts.append(f"Order Details: {order_response}")
        
        # Include summary if present
        if summary_response:
            final_parts.append(f"Summary: {summary_response}")
        
        # Add flow execution metadata
        final_parts.append(f"\n[Flow Execution: Intent={flow_responses.get('intent', 'unknown')} | Flows Used: {', '.join(flow_responses.keys())}]")
        
        final_response = "\n\n".join(final_parts)
        
        return final_response
        
    except Exception as e:
        print(f"Error in end-to-end processing: {e}")
        # Fallback to simple response
        return f"I apologize, but I'm having trouble processing your request. As a {restaurant_brand} assistant, I'm here to help with your order. Could you please try again?\n\n[Error: {str(e)}]"

In [8]:
from azure.ai.evaluation.simulator import Simulator

async def ordering_simulator_callback(
    messages: List[Dict],
    stream: bool = False,
    session_state: Optional[str] = None,
    context: Optional[Dict[str, Any]] = None,
) -> dict:
    """
    Enhanced simulator callback for end-to-end ordering evaluation.
    Maintains order state and context across the conversation.
    """
    messages_list = messages["messages"]
    latest_message = messages_list[-1]
    
    # Extract query and context
    user_query = latest_message["content"]
    grounding_context = latest_message.get("context", "")
    
    # Maintain session state for order continuity
    if session_state:
        try:
            order_state = json.loads(session_state)
        except:
            order_state = {"items": [], "total": 0.0}
    else:
        order_state = {"items": [], "total": 0.0}
    
    # Call the end-to-end ordering application
    try:
        response = await end_to_end_ordering_application(user_query, grounding_context)
        
        # Update session state (simulate order persistence)
        updated_session_state = json.dumps(order_state)
        
        # Format response in OpenAI chat protocol
        message = {
            "content": response,
            "role": "assistant", 
            "context": grounding_context,
            "order_state": order_state,
            "evaluation_metadata": {
                "brand": restaurant_brand,
                "style": conversation_style,
                "intent_processed": True,
                "order_updated": len(order_state["items"]) > 0
            }
        }
        
        messages["messages"].append(message)
        
        return {
            "messages": messages["messages"],
            "stream": stream,
            "session_state": updated_session_state,
            "context": grounding_context
        }
        
    except Exception as e:
        print(f"Error in ordering simulator callback: {e}")
        # Return error response
        error_message = {
            "content": f"I'm sorry, I encountered an error processing your request. Please try again.",
            "role": "assistant",
            "context": grounding_context,
            "error": str(e)
        }
        messages["messages"].append(error_message)
        return {"messages": messages["messages"], "stream": stream, "session_state": session_state, "context": context}


In [9]:
# Initialize the ordering-specific simulator

azure_model_config = AzureOpenAIModelConfiguration(
	azure_endpoint=str(model_config["azure_endpoint"]),
	azure_deployment=str(model_config["azure_deployment"]),
	api_key=str(model_config["api_key"]),
	api_version=str(model_config.get("api_version", "2024-12-01-preview"))
)

ordering_simulator = Simulator(model_config=azure_model_config)
print("Ordering simulator initialized successfully")

Class Simulator: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.


Ordering simulator initialized successfully


In [10]:
# Run the simulation with ordering-specific scenarios
print(f"Running simulation with {len(conversation_turns)} ordering scenarios...")

outputs = await ordering_simulator(
    target=ordering_simulator_callback,
    conversation_turns=conversation_turns,
    max_conversation_turns=1,
    concurrent_async_tasks=2  # Reduced for stability with complex flows
)

print(f"Simulation completed. Generated {len(outputs)} response pairs.")


Running simulation with 6 ordering scenarios...


Simulating with predefined conversation turns:   0%|                    | 0/6 [00:00<?, ?messages/s]Invalid item for contoso_restaurant: burger, quarter lb, single, sesame, normal
Invalid item for contoso_restaurant: burger, quarter lb, single, sesame, normal
Simulating with predefined conversation turns:  33%|████        | 2/6 [00:27<00:53, 13.45s/messages]Invalid item for contoso_restaurant: sprite
Invalid item for contoso_restaurant: sprite
Simulating with predefined conversation turns:  50%|██████      | 3/6 [01:15<01:26, 28.87s/messages]Item 'burrito' not found in contoso_restaurant menu configuration. Using generic type.
Item 'vegetable spring roll' not found in contoso_restaurant menu configuration. Using generic type.
Item 'burrito' not found in contoso_restaurant menu configuration. Using generic type.
Item 'vegetable spring roll' not found in contoso_restaurant menu configuration. Using generic type.
Simulating with predefined conversation turns:  67%|████████    | 4/6 [01:21

Simulation completed. Generated 6 response pairs.


In [11]:
# Convert outputs to evaluation format
output_file = "ordering_groundedness_evaluation.jsonl"
with Path(output_file).open("w") as file:
    for output in outputs:
        file.write(output.to_eval_qr_json_lines())

print(f"Evaluation data saved to: {output_file}")

# Preview the evaluation data
with open(output_file, "r") as f:
    sample_line = f.readline()
    sample_data = json.loads(sample_line) if sample_line else {}
    print("Sample evaluation data:")
    print(json.dumps(sample_data, indent=2) + "..." if len(str(sample_data)) > 500 else json.dumps(sample_data, indent=2))


Evaluation data saved to: ordering_groundedness_evaluation.jsonl
Sample evaluation data:
{
  "query": "What menu items do you have?",
  "response": "Response: Welcome to Contoso Restaurant! We\u2019re delighted to share our menu filled with delicious, home-style offerings made with love and fresh, organic ingredients. Here\u2019s what we\u2019ve got:\n\n### **Burgers**  \n- **Hamburger** (also known as \"burger\")  \n- **Cheeseburger** (also called \"cheese burger\")  \n- **Black Bean Burger** (our veggie-friendly option, also known as \"veggie burger\" or \"vegetarian burger\")\n\n### **Sides**  \n- **Fries** (or \"french fries\")\n\n### **Drinks**  \n- **Cola** (you might call it \"coke\" or \"coca cola\")  \n- **Diet Cola** (also referred to as \"diet coke\" or simply \"diet\")  \n- **Lemon-Lime Soda** (like \"sprite\" or \"lemon lime soda\")  \n- **Root Beer**\n\n### **Toppings for Burgers:**  \nWe offer plenty of options so you can customize your burger just the way you like it:  

In [3]:
import json
from pathlib import Path

def read_jsonl_file(file_path: Path) -> list:
    data = []
    try:
        with open(file_path, 'r') as f:
            for line in f:
                if line.strip():  # Skip empty lines
                    try:
                        json_obj = json.loads(line)  # Parse each line as separate JSON
                        data.append(json_obj)
                    except json.JSONDecodeError as e:
                        print(f"Error parsing line: {e}")
                        continue
        return data
    except FileNotFoundError:
        print(f"File not found: {file_path}")
        return []
    except Exception as e:
        print(f"Error reading file: {e}")
        return []

# Read the JSONL file
output_file = Path("ordering_groundedness_evaluation.jsonl")
data = read_jsonl_file(output_file)

In [4]:
from azure.ai.evaluation import GroundednessEvaluator, RelevanceEvaluator, CoherenceEvaluator, FluencyEvaluator, evaluate

azure_model_config = AzureOpenAIModelConfiguration(
	azure_endpoint=str(model_config["azure_endpoint"]),
	azure_deployment=str(model_config["azure_deployment"]),
	api_key=str(model_config["api_key"]),
	api_version=str(model_config["api_version"])
)

groundedness_evaluator = GroundednessEvaluator(model_config=azure_model_config, max_completion_tokens=3000)
relevance_evaluator = RelevanceEvaluator(model_config=azure_model_config)
coherence_evaluator = CoherenceEvaluator(model_config=azure_model_config)
fluency_evaluator = FluencyEvaluator(model_config=azure_model_config)

eval_output2 = evaluate(
    data=output_file,  
    evaluators={
        "groundedness": groundedness_evaluator,
        "relevance": relevance_evaluator,
        "coherence": coherence_evaluator,
        "fluency": fluency_evaluator,
    },
    max_completion_tokens=3000
)
print(eval_output2)

[2025-07-30 11:01:22 -0700][promptflow._core.entry_meta_generator][WARNING] - Generate meta in current process and timeout won't take effect. Please handle timeout manually outside current process.
[2025-07-30 11:01:22 -0700][promptflow._core.entry_meta_generator][WARNING] - Generate meta in current process and timeout won't take effect. Please handle timeout manually outside current process.
[2025-07-30 11:01:22 -0700][promptflow._core.entry_meta_generator][WARNING] - Generate meta in current process and timeout won't take effect. Please handle timeout manually outside current process.
[2025-07-30 11:01:22 -0700][promptflow._core.entry_meta_generator][WARNING] - Generate meta in current process and timeout won't take effect. Please handle timeout manually outside current process.
[2025-07-30 11:01:22 -0700][promptflow._sdk._orchestrator.run_submitter][INFO] - Submitting run azure_ai_evaluation_evaluators_groundedness_20250730_110120_755617, log path: C:\Users\t-toluale\.promptflow\.ru

2025-07-30 11:01:22 -0700   58232 execution.bulk     INFO     Current thread is not main thread, skip signal handler registration in BatchEngine.
2025-07-30 11:01:28 -0700   58232 execution.bulk     INFO     Finished 1 / 6 lines.
2025-07-30 11:01:28 -0700   58232 execution.bulk     INFO     Average execution time for completed lines: 6.28 seconds. Estimated time for incomplete lines: 31.4 seconds.
2025-07-30 11:01:28 -0700   58232 execution.bulk     INFO     Finished 2 / 6 lines.
2025-07-30 11:01:28 -0700   58232 execution.bulk     INFO     Average execution time for completed lines: 3.18 seconds. Estimated time for incomplete lines: 12.72 seconds.
2025-07-30 11:01:28 -0700   58232 execution.bulk     INFO     Finished 3 / 6 lines.
2025-07-30 11:01:28 -0700   58232 execution.bulk     INFO     Average execution time for completed lines: 2.14 seconds. Estimated time for incomplete lines: 6.42 seconds.
2025-07-30 11:01:28 -0700   58232 execution.bulk     INFO     Finished 4 / 6 lines.
2025

In [5]:
import json
from datetime import datetime
from pathlib import Path

def extract_evaluation_data(data):
    """Extract key information from evaluation results and format it for JSON."""
    extracted_data = {
        "conversations": []
    }

    for row in data["rows"]:
        conversation = {
            "query": row["inputs.query"],
            "response": row["inputs.response"],
            "metrics": {
                "groundedness": {
                    "score": row["outputs.groundedness.groundedness"],
                    "reason": row["outputs.groundedness.groundedness_reason"]
                },
                "relevance": {
                    "score": row["outputs.relevance.relevance"],
                    "reason": row["outputs.relevance.relevance_reason"]
                },
                "coherence": {
                    "score": row["outputs.coherence.coherence"],
                    "reason": row["outputs.coherence.coherence_reason"]
                },
                "fluency": {
                    "score": row["outputs.fluency.fluency"],
                    "reason": row["outputs.fluency.fluency_reason"]
                }
            }
        }
        extracted_data["conversations"].append(conversation)

    return extracted_data

def save_evaluation_results(data, filename="gpt-4.1-end-to-end_evaluation_results.json"):
    """Save the extracted evaluation data to a JSON file."""
    try:
        # Extract the key information
        extracted_data = extract_evaluation_data(data)
        
        # Create output directory if it doesn't exist
        output_dir = Path("overall_evaluation_results")
        output_dir.mkdir(exist_ok=True)
        
        # Generate timestamped filename
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        output_file = output_dir / f"{timestamp}_{filename}"
        
        # Save to JSON file with pretty printing
        with open(output_file, 'w', encoding='utf-8') as f:
            json.dump(extracted_data, f, indent=2, ensure_ascii=False)
            
        print(f"Results successfully saved to: {output_file}")
        return output_file
        
    except Exception as e:
        print(f"Error saving results: {e}")
        return None


if eval_output2 is not None:
    output_file = save_evaluation_results(eval_output2)


In [ ]:
import json
import asyncio
from pathlib import Path
from openai import AzureOpenAI
from typing import List, Dict, Any
import os
from datetime import datetime

async def LLM_as_judge(query: str, response: str, context: str, criteria: List[str]) -> Dict[str, Any]:
    """
    Evaluate restaurant chatbot response using o1-mini as judge.
    Returns structured evaluation with scores 1-5 and reasoning.
    """
    criteria_text = ", ".join(criteria)
    
    evaluation_prompt = f"""
    You are an expert evaluator for restaurant chatbot responses. Evaluate the following response based on the specified criteria.
    
    QUERY: {query}
    
    RESPONSE: {response}
    
    CONTEXT: {context}
    
    EVALUATION CRITERIA: {criteria_text}
    
    Rate each criterion on a scale of 1-5 where:
    - 1 = Very Poor
    - 2 = Poor  
    - 3 = Average
    - 4 = Good
    - 5 = Excellent
    
    Evaluate these specific criteria:
    - Groundedness: How factually accurate is the response based on the provided context? Does it stay within menu boundaries? Are menu customization accurately captured?
    - Relevance: How well does the response address the user's specific query and intent?
    - Coherence: Is the response logically structured, clear, and easy to follow?
    - Fluency: Is the response language natural, grammatically correct, and well written?
    
    IMPORTANT: Return your evaluation as a valid JSON object with this exact structure:
    {{
        "groundedness": {{
            "score": <1-5>,
            "reasoning": "Detailed explanation for the groundedness score"
        }},
        "relevance": {{
            "score": <1-5>,
            "reasoning": "Detailed explanation for the relevance score"
        }},
        "coherence": {{
            "score": <1-5>,
            "reasoning": "Detailed explanation for the coherence score"
        }},
        "fluency": {{
            "score": <1-5>,
            "reasoning": "Detailed explanation for the fluency score"
        }},
        "overall_score": <average of all scores>,
        "summary": "Brief overall assessment of the response quality"
    }}
    """
    
    client = AzureOpenAI(
        api_key=api_key,
        api_version="2024-12-01-preview",
        azure_endpoint=endpoint
    )
    
    try:
        response_obj = client.chat.completions.create(
            model=deployment_name,  
            messages=[{"role": "user", "content": evaluation_prompt}],
            max_completion_tokens=3000
        )
        
        evaluation_result = response_obj.choices[0].message.content
        
        # Try to parse JSON response
        try:
            return json.loads(evaluation_result)
        except json.JSONDecodeError:
            # If JSON parsing fails, return a structured error
            return {
                "error": "Failed to parse evaluation JSON"}
            
    except Exception as e:
        return {
            "error": f"API call failed: {str(e)}"}

In [ ]:
async def evaluate_jsonl_file(file_path: str, output_path: str = None) -> List[Dict[str, Any]]:
    """
    Evaluate all responses in a JSONL file and save results.
    """
    # Load your environment variables
    api_key = os.getenv("AZURE_OPENAI_API_KEY")
    endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
    deployment_name = os.getenv("AZURE_OPENAI_DEPLOYMENT_NAME")
    
    if not api_key or not endpoint:
        raise ValueError("Missing Azure OpenAI credentials in environment variables")
    
    results = []
    criteria = ["Groundedness", "Relevance", "Coherence", "Fluency"]
    
    # Read JSONL file
    with open(file_path, 'r', encoding='utf-8') as f:
        for line_num, line in enumerate(f, 1):
            if line.strip():
                try:
                    data = json.loads(line)
                    query = data.get('query', '')
                    response = data.get('response', '')
                    context = data.get('context', '')
                    
                    print(f"Evaluating entry {line_num}...")
                    
                    # Evaluate the response
                    evaluation = await LLM_as_judge(query, response, context, criteria)
                    
                    # Combine original data with evaluation
                    result = {
                        "line_number": line_num,
                        "original_data": data,
                        "evaluation": evaluation,
                        "timestamp": datetime.now().isoformat()
                    }
                    
                    results.append(result)
                    
                    # Add small delay to avoid rate limiting
                    await asyncio.sleep(1)
                    
                except json.JSONDecodeError as e:
                    print(f"Error parsing line {line_num}: {e}")
                    continue
                except Exception as e:
                    print(f"Error evaluating line {line_num}: {e}")
                    continue
    
    # Save results
    if output_path is None:
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        output_path = f"evaluation_results_o3-mini_{timestamp}.json"
    
    with open(output_path, 'w', encoding='utf-8') as f:
        json.dump(results, f, indent=2, ensure_ascii=False)
    
    print(f"Evaluation complete! Results saved to: {output_path}")
    print(f"Evaluated {len(results)} entries")
    
    return results

async def run_evaluation():
    """
    Main function to run the evaluation on your JSONL file.
    """
    input_file = "ordering_groundedness_evaluation.jsonl"
    
    # Check if file exists
    if not Path(input_file).exists():
        print(f"File {input_file} not found in current directory")
        return
    
    try:
        results = await evaluate_jsonl_file(input_file)
        
        # Print summary statistics
        if results:
            scores = []
            for result in results:
                if 'evaluation' in result and 'overall_score' in result['evaluation']:
                    scores.append(result['evaluation']['overall_score'])
            
            if scores:
                avg_score = sum(scores) / len(scores)
                print(f"\nSummary Statistics:")
                print(f"Average Overall Score: {avg_score:.2f}")
                print(f"Highest Score: {max(scores):.2f}")
                print(f"Lowest Score: {min(scores):.2f}")
        
    except Exception as e:
        print(f"Error during evaluation: {e}")

# Usage example
if __name__ == "__main__":
    # Run the evaluation
    await run_evaluation()

Evaluating entry 1...
Evaluating entry 2...
Evaluating entry 3...
Evaluating entry 4...
Evaluating entry 5...
Evaluating entry 6...
Evaluation complete! Results saved to: evaluation_results_o3-mini_20250730_105453.json
Evaluated 6 entries

Summary Statistics:
Average Overall Score: 4.62
Highest Score: 5.00
Lowest Score: 4.00
